In [ ]:
import pandas as pd

df = pd.read_json('data/review-Alabama_10.json', lines=True)
df

c:\Users\tanwe\anaconda3\Lib\site-packages\pandas\core\dtypes\astype.py:189: RuntimeWarning: invalid value encountered in cast
  return values.astype(dtype, copy=copy)


,user_id,name,time,rating,text,pics,resp,gmap_id
0,1.140438e+20,Kanisha Mixon,1597168272670,5,Very Personable staff! Beautiful and clean env...,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
1,1.160090e+20,Brandie Hodges,1609899039594,5,Best clothing intown,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
2,1.062399e+20,Sharon King,1547235290843,4,None,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
3,1.049701e+20,Veronica Pierce,1517709403534,5,None,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
4,1.105875e+20,Whitney Waldon Collier,1535245718492,5,None,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
...,...,...,...,...,...,...,...,...
5146325,1.135535e+20,Tahniyath Sultana,1575316959599,5,None,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5146326,1.149057e+20,Cody Mc,1541393753107,5,None,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5146327,1.146100e+20,Liam Wood,1499539724361,5,None,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5146328,1.054581e+20,John Laine,1506555396467,3,None,None,None,0x888912c75384e095:0x3bf8b383c85ccf97


In [4]:
df = df.dropna(subset=['text'])
df

,user_id,name,time,rating,text,pics,resp,gmap_id
0,1.140438e+20,Kanisha Mixon,1597168272670,5,Very Personable staff! Beautiful and clean env...,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
1,1.160090e+20,Brandie Hodges,1609899039594,5,Best clothing intown,None,None,0x8862134e67ff5c87:0x38b5e2ae99cd1fcf
5,1.120426e+20,Emily Miles,1611850938780,1,"Not friendly at all, as I ask questions about ...",None,None,0x886268e8fdc4fd2f:0x746533eb9aa4d4df
6,1.089190e+20,Faye Ahzburjn,1516515504358,4,They have beautiful baby and children's clothi...,None,None,0x886268e8fdc4fd2f:0x746533eb9aa4d4df
7,1.018531e+20,Amber Winn,1562178900806,3,"Cute shop, but the lack of boy clothes is sad....",None,None,0x886268e8fdc4fd2f:0x746533eb9aa4d4df
...,...,...,...,...,...,...,...,...
5144376,1.015225e+20,Brenda Vason,1532211048663,5,(Translated by Google) Visited Boys\n\n(Origin...,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5144377,1.027384e+20,Jacob Mcnair,1505851499086,5,(Translated by Google) Me\n\n(Original)\nYo,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5144378,1.077699e+20,Sidnei Geraldo,1490288825090,5,(Translated by Google) Many brands and unbeata...,None,None,0x888912c75384e095:0x3bf8b383c85ccf97
5144379,1.046066e+20,AX -1,1502075756081,3,(Translated by Google) Not very cheap except f...,None,None,0x888912c75384e095:0x3bf8b383c85ccf97


In [ ]:
from __future__ import annotations
import json, re, math
import torch
import pandas as pd
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cpu",              # CPU only
    torch_dtype=torch.float32,     # CPU-friendly; BF16 requires AVX512 on some CPUs
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

# Qwen chat models prefer left padding for batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

: 

In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv()
# azure_openai_key = os.getenv('AZURE_OPENAI_API_KEY_3')
# azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT_3')
# azure_openai_version = os.getenv('AZURE_OPENAI_VERSION') 
# azure_openai_deployment = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME_GPT4o')

# print(f"API Key loaded: {'Yes' if azure_openai_key else 'No'}")
# print(f"Endpoint loaded: {'Yes' if azure_openai_endpoint else 'No'}")

API Key loaded: Yes
Endpoint loaded: Yes


In [ ]:
# from openai import AzureOpenAI

# azure_model = AzureOpenAI(
#     api_version=azure_openai_version,
#     azure_endpoint=azure_openai_endpoint,
#     api_key=azure_openai_key,
#     azure_deployment=azure_openai_deployment,
#     temperature=0
# )

In [ ]:
LABELS = ["NO_AD", "IRRELEVANT", "RANT_NO_VISIT"]

POLICY_TEXT = (
    "Policy (apply strictly):\n"
    "- NO_AD (No Advertisement): Reviews must not contain promotional content, discount codes, referral links, or calls to visit external sites.\n"
    "- IRRELEVANT (No Irrelevant Content): The review must be about the location/business experience (service, products, environment). Unrelated topics are violations.\n"
    "- RANT_NO_VISIT (No Rant Without Visit): Complaints/rants must come from actual visitors; hearsay or explicit 'never been' is a violation.\n"
)

FEW_SHOTS = """
Review: "Best pizza! Visit www.pizzapromo.com for discounts!"
Assistant: {"labels":["NO_AD"], "rationale":"Contains promotional link", "confidence":0.90}

Review: "Amazing food and great service. Highly recommend this place!"
Assistant: {"labels":[], "rationale":"Genuine recommendation without promotional content", "confidence":0.92}

Review: "Check out my blog at foodreviews.com for more reviews!"
Assistant: {"labels":["NO_AD"], "rationale":"Contains promotional link to personal blog", "confidence":0.90}

Review: "I love my new phone, but this place is too noisy."
Assistant: {"labels":["IRRELEVANT"], "rationale":"Review contains irrelevant content about phone", "confidence":0.90}

Review: "The atmosphere was perfect for a quiet dinner. Food was delicious."
Assistant: {"labels":[], "rationale":"Review focuses on the restaurant/business experience", "confidence":0.92}

Review: "Best clothing intown"
Assistant: {"labels":[], "rationale":"Review is about the clothing store", "confidence":0.92}

Review: "Never been here, but I heard it's terrible."
Assistant: {"labels":["RANT_NO_VISIT"], "rationale":"Reviewer explicitly states they haven't visited", "confidence":0.90}

Review: "Visited last week and was disappointed with the slow service."
Assistant: {"labels":[], "rationale":"Reviewer indicates they actually visited", "confidence":0.92}

Review: "Not friendly at all, as I ask questions about the clothes"
Assistant: {"labels":[], "rationale":"Reviewer implies they were physically present asking questions", "confidence":0.92}
"""

INSTR = (
    "Instructions:\n"
    "- Decide labels from: NO_AD, IRRELEVANT, RANT_NO_VISIT (multi-label allowed).\n"
    "- If none apply, return \"labels\": [] (client will mark COMPLIANT).\n"
    "- Return ONE JSON object only with fields: labels, rationale, confidence (0.0–1.0). "
    "No extra text.\n"
    "- Keep rationale ≤ 2 sentences."
)

SYSTEM_MSG = (
    "You are a strict policy labeler for location reviews. Apply the policy exactly. "
    "Always return valid JSON as specified."
)


In [ ]:
def build_messages_for_review(review_text):

    user_prompt = (
        f"{POLICY_TEXT}\n{INSTR}\n\nFew-shot examples:\n{FEW_SHOTS}\n\n"
        f"Now classify this review. Return one JSON object only.\n"
        f"Review: {review_text}\nAssistant:"
    )
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_prompt},
    ]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B-Instruct-2507")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-30B-A3B-Instruct-2507")

inputs = tokenizer.apply_chat_template(
	df['text'],
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [6]:
df1 = pd.read_json('data/review-Washington_10.json', lines=True)
df1

ValueError: Could not reserve memory block

In [ ]:
df1 = df1.dropna(subset=['text'])
df1